In [ ]:
!pip install black==24.3.0 --no-deps
!pip install pylint==2.17.7 --no-deps

# Configuracion

In [2]:
team = 'RIESGOS' # TODO 1: Colocar nombre del equipo: RETAIL, RIESGOS
name_ds = 'Zelaya Contreras Jesus' # TODO 2: Colocar mis apellidos y nombres
path_script = './inference.py'

In [ ]:
import os
import boto3
from sagemaker import get_execution_role

account = boto3.client('sts').get_caller_identity()['Account']
#image_uri = f'{account}.dkr.ecr.us-east-1.amazonaws.com/sagemaker-python3:3.8.15-cpu'
image_uri = f'{account}.dkr.ecr.us-east-1.amazonaws.com/sagemaker-python3:3.10.14-cpu' #entrene con esta version 3.12.13
role_arn = get_execution_role()

os.environ['I_TEAM_RETAIL'], os.environ['I_CC_RETAIL'] = 'DS RETAIL', '9946100000'
os.environ['I_TEAM_RIESGOS'], os.environ['I_CC_RIESGOS'] = 'DS RIESGOS', '9810200000'

tags = [
    {'Key': 'I_RESPONSABLE_LT', 'Value': name_ds},
    {'Key': 'I_APLICACION', 'Value': 'SDLF'},
    {'Key': 'I_PROYECTO', 'Value': 'SDLF'},
    {'Key': 'I_AMBIENTE', 'Value': 'DEV'},
    {'Key': 'I_CUENTA', 'Value': account},
    {'Key': 'I_SIGLA', 'Value': 'SAN'},
    {'Key': 'I_TEAM', 'Value': os.environ[f'I_TEAM_{team}']},
    {'Key': 'I_CC', 'Value': os.environ[f'I_CC_{team}']},
]

# Formatear Script

In [ ]:
!black --line-length 100 $path_script
!pylint --disable C0103 $path_script

# Crear Processor

| Instancia      | CPU Cores | Memoria (GB) |
|----------------|-----------|--------------|
| ml.m5.large    | 2         | 8            |
| ml.m5.xlarge   | 4         | 16           |
| ml.m5.2xlarge  | 8         | 32           |
| ml.m5.4xlarge  | 16        | 64           |
| ml.m5.12xlarge | 48        | 192          |
| ml.m5.24xlarge | 96        | 384          |

In [ ]:
from sagemaker.processing import ScriptProcessor

processor = ScriptProcessor(instance_type='ml.m5.2xlarge',
                            volume_size_in_gb=30,
                            instance_count=1,
                            command=['python3'],
                            image_uri=image_uri,
                            role=role_arn,
                            tags=tags)

# Definir Entradas

In [6]:
from sagemaker.processing import ProcessingInput

inputs = []

inputs.append(ProcessingInput(
    input_name='code/utils',
    source='s3://ibk-discovery-riesgos-us-east-1-339712995012-data/discovery/riesgos/B39918/2026/reentreno_pp_2026/envio_mlops/utils/', # TODO 3: Colocar ubicacion de S3 del script 'utils.py'
    destination='/opt/ml/processing/input/code/utils',
))

inputs.append(ProcessingInput(
    input_name='data',
    source='s3://ibk-discovery-riesgos-us-east-1-339712995012-data/discovery/riesgos/B39918/2026/reentreno_pp_2026/base/ds_cppibkren_mdl/p_codmes=202606/', # TODO 4: Colocar ubicacion de S3 de los datos del periodo de inferencia  # Add Calibration: apuntar al ds RECONSTRUIDO con el inference.sql calibrado (trae trf_tipo_ingreso_dep/trf_macro01/trf_macro02)
    destination='/opt/ml/processing/input/data',
))

inputs.append(ProcessingInput(
    input_name='models',
    source='s3://ibk-discovery-riesgos-us-east-1-339712995012-data/discovery/riesgos/B39918/2026/reentreno_pp_2026/envio_mlops/models/', # TODO 5: Colocar ubicacion de S3 de los modelos entrenados  # Add Calibration: el prefijo models/ debe incluir calibracion_ree.json ademas del .pkl
    destination='/opt/ml/processing/input/models',
))

# Definir Salidas

In [7]:
from sagemaker.processing import ProcessingOutput

outputs = []

outputs.append(ProcessingOutput(
    output_name='results',
    source='/opt/ml/processing/output/results',
    destination='s3://ibk-discovery-riesgos-us-east-1-339712995012-data/discovery/riesgos/B39918/2026/reentreno_pp_2026/envio_mlops/results/', # TODO 6: Colocar ubicacion de S3 donde van a estar los resultados
))

# Ejecutar Job

In [ ]:
model = 'cppibkren' # TODO 7: Colocar nombre de la rama del modelo
partition = '202606' # TODO 8: Colocar periodo de los datos DEBE IR CON 01 AL FINAL?

arguments = [
    '--model', model,
    '--table-score', f'scr_{model}',
    '--partition', partition,
]

processor.run(code=path_script,
              inputs=inputs,
              outputs=outputs,
              arguments=arguments)

### Bibliotecas Instaladas

- `image_uri = f'{account}.dkr.ecr.us-east-1.amazonaws.com/sagemaker-python3:3.10.14-cpu'`  

    ```
    aiohappyeyeballs          2.6.1
    aiohttp                   3.13.2
    aiosignal                 1.4.0
    alabaster                 1.0.0
    annotated-doc             0.0.4
    annotated-types           0.7.0
    antlr4-python3-runtime    4.9.3
    anyio                     4.11.0
    argon2-cffi               25.1.0
    argon2-cffi-bindings      25.1.0
    arrow                     1.4.0
    astroid                   4.0.1
    astropy                   6.1.7
    astropy-iers-data         0.2025.11.3.0.38.37
    async-lru                 2.0.5
    async-timeout             5.0.1
    atomicwrites              1.4.1
    attrs                     25.4.0
    Authlib                   1.6.5
    autopep8                  1.6.0
    autovizwidget             0.23.0
    awscli                    1.42.72
    babel                     2.17.0
    backcall                  0.2.0
    backports-datetime-fromisoformat 2.0.3
    backports.tarfile         1.2.0
    beautifulsoup4            4.14.2
    binaryornot               0.4.4
    bitarray                  3.7.2
    black                     24.3.0
    bleach                    6.2.0
    blinker                   1.9.0
    bokeh                     3.8.0
    boto3                     1.40.72
    botocore                  1.40.72
    Bottleneck                1.6.0
    Brotli                    1.2.0
    brotlipy                  0.7.0
    cached-property           1.5.2
    certifi                   2025.10.5
    cffi                      2.0.0
    chardet                   5.2.0
    charset-normalizer        3.4.4
    click                     8.3.0
    cloudpickle               3.1.2
    colorama                  0.4.6
    comm                      0.2.3
    contourpy                 1.3.2
    cookiecutter              2.6.0
    coverage                  7.11.0
    cryptography              46.0.3
    cycler                    0.12.1
    Cython                    3.1.6
    cytoolz                   1.1.0
    dask                      2025.10.0
    debugpy                   1.8.17
    decorator                 5.2.1
    defusedxml                0.7.1
    diff-match-patch          20241021
    dill                      0.4.0
    distributed               2025.10.0
    docker                    7.1.0
    docutils                  0.19
    dparse                    0.6.4
    entrypoints               0.4
    et_xmlfile                2.0.0
    exceptiongroup            1.3.0
    fastapi                   0.121.1
    fastcache                 1.1.0
    fastjsonschema            2.21.2
    filelock                  3.20.0
    flake8                    4.0.1
    Flask                     3.1.2
    flask-cors                6.0.1
    fonttools                 4.60.1
    fqdn                      1.5.1
    frozenlist                1.7.0
    fsspec                    2025.10.0
    future                    1.0.0
    gevent                    25.9.1
    gmpy2                     2.2.1
    google-pasta              0.2.0
    graphene                  3.4.3
    graphql-core              3.2.7
    graphql-relay             3.2.0
    greenlet                  3.2.4
    gssapi                    1.10.1
    h11                       0.16.0
    h2                        4.3.0
    h5py                      3.13.0
    hdijupyterutils           0.23.0
    hpack                     4.1.0
    hyperframe                6.1.0
    idna                      3.11
    imagecodecs-lite          2019.12.3
    imageio                   2.37.0
    imagesize                 1.4.1
    immutables                0.21
    importlib-metadata        6.11.0
    importlib_resources       6.5.2
    inflection                0.5.1
    iniconfig                 2.3.0
    intervaltree              3.1.0
    ipykernel                 7.1.0
    ipython                   7.33.0
    ipython_genutils          0.2.0
    ipywidgets                8.1.8
    isoduration               20.11.0
    isort                     7.0.0
    itsdangerous              2.2.0
    jaraco.classes            3.4.0
    jaraco.context            6.0.1
    jaraco.functools          4.3.0
    jedi                      0.18.2
    jeepney                   0.9.0
    jellyfish                 1.1.3
    Jinja2                    3.1.6
    jmespath                  1.0.1
    joblib                    1.5.2
    json5                     0.12.1
    jsonpointer               3.0.0
    jsonschema                4.25.1
    jsonschema-specifications  2025.9.1
    jupyter                   1.1.1
    jupyter_client            8.6.3
    jupyter-console           6.6.3
    jupyter_core              5.9.1
    jupyter-events            0.12.0
    jupyter-lsp               2.3.0
    jupyter_server            2.17.0
    jupyter_server_terminals  0.5.3
    jupyterlab                4.4.10
    jupyterlab_pygments       0.3.0
    jupyterlab_server         2.28.0
    jupyterlab_widgets        3.0.16
    keyring                   25.6.0
    kiwisolver                1.4.9
    krb5                      0.8.0
    lark                      1.3.1
    lazy_loader               0.4
    lazy-object-proxy         1.12.0
    lief                      0.17.0
    llvmlite                  0.45.1
    locket                    1.0.0
    lz4                       4.4.4
    markdown-it-py            4.0.0
    MarkupSafe                3.0.3
    marshmallow               4.1.0
    matplotlib                3.10.7
    matplotlib-inline         0.2.1
    mccabe                    0.6.1
    mdurl                     0.1.2
    mistune                   3.1.4
    mkl_fft                   2.0.0
    mkl-service               2.5.2
    mock                      4.0.3
    more-itertools            10.8.0
    mpmath                    1.3.0
    msgpack                   1.1.2
    multidict                 6.6.3
    multiprocess              0.70.18
    munkres                   1.1.4
    mypy_extensions           1.1.0
    narwhals                  2.10.2
    nbclient                  0.10.2
    nbconvert                 7.16.6
    nbformat                  5.10.4
    nest_asyncio              1.6.0
    networkx                  3.4.2
    nltk                      3.9.2
    notebook                  7.4.7
    notebook_shim             0.2.4
    numba                     0.62.1
    numexpr                   2.14.1
    numpy                     1.26.4
    numpydoc                  1.9.0
    nvidia-ml-py              13.580.82
    omegaconf                 2.3.0
    openpyxl                  3.1.5
    overrides                 7.7.0
    packaging                 24.2
    pandas                    2.3.3
    pandocfilters             1.5.0
    parso                     0.8.5
    partd                     1.4.2
    path                      17.1.1
    pathlib2                  2.3.7.post1
    pathos                    0.3.4
    pathspec                  0.12.1
    patsy                     1.0.2
    pexpect                   4.9.0
    pickleshare               0.7.5
    pillow                    12.0.0
    pip                       25.2
    pkginfo                   1.12.1.2
    platformdirs              4.5.0
    plotly                    6.3.1
    pluggy                    1.6.0
    ply                       3.11
    pox                       0.3.6
    ppft                      1.7.7
    prometheus_client         0.23.1
    prompt_toolkit            3.0.52
    propcache                 0.3.1
    protobuf                  6.31.1
    psutil                    7.1.3
    psycopg2                  2.9.10
    psycopg2-binary           2.9.10
    ptyprocess                0.7.0
    py-cpuinfo                9.0.0
    py4j                      0.10.9.5
    pyarrow                   22.0.0
    pyasn1                    0.6.1
    pycodestyle               2.8.0
    pycosat                   0.6.6
    pycparser                 2.22
    pycryptodome              3.23.0
    pycurl                    7.45.7
    pydantic                  2.12.3
    pydantic_core             2.41.4
    pydocstyle                6.3.0
    pyerfa                    2.0.1.5
    pyflakes                  2.4.0
    Pygments                  2.19.2
    pykerberos                1.2.4
    pylint                    2.17.7
    pyls-spyder               0.4.0
    pynvml                    13.0.1
    pyodbc                    5.3.0
    pyOpenSSL                 25.3.0
    pyparsing                 3.2.5
    PyQt5                     5.15.11
    PyQt5_sip                 12.17.0
    pyrsistent                0.20.0
    PySide6                   6.9.3
    PySocks                   1.7.1
    pyspark                   3.3.0
    pyspnego                  0.11.2
    pytest                    8.4.2
    python-dateutil           2.9.0.post0
    python-json-logger        2.0.7
    python-lsp-black          1.1.0
    python-lsp-jsonrpc        1.1.2
    python-lsp-server         1.3.3
    python-slugify            8.0.4
    pytoolconfig              1.2.5
    pytz                      2025.2
    PyWavelets                1.8.0
    pyxdg                     0.28
    PyYAML                    6.0.3
    pyzmq                     27.1.0
    QDarkStyle                3.0.2
    qstylizer                 0.2.4
    QtAwesome                 1.4.0
    qtconsole                 5.2.2
    QtPy                      2.4.3
    referencing               0.37.0
    regex                     2025.11.3
    requests                  2.32.5
    requests-kerberos         0.15.0
    rfc3339_validator         0.1.4
    rfc3986-validator         0.1.1
    rfc3987-syntax            1.1.0
    rich                      14.2.0
    rope                      1.14.0
    rpds-py                   0.28.0
    rsa                       4.7.2
    rtree                     1.4.1
    ruamel.yaml               0.18.16
    ruamel.yaml.clib          0.2.14
    ruamel-yaml-conda         0.15.80
    s3fs                      0.4.2
    s3transfer                0.14.0
    safety-schemas            0.0.16
    sagemaker                 2.254.1
    sagemaker-core            1.0.64
    sagemaker_pyspark         1.4.5
    schema                    0.7.8
    scikit-image              0.20.0
    scikit-learn              1.7.2
    scipy                     1.15.2
    seaborn                   0.13.2
    SecretStorage             3.4.0
    Send2Trash                1.8.3
    setuptools                80.9.0
    shap                      0.48.0
    shellingham               1.5.4
    shiboken6                 6.9.3
    sip                       6.10.0
    six                       1.17.0
    slicer                    0.0.8
    smdebug-rulesconfig       1.0.1
    sniffio                   1.3.1
    snowballstemmer           3.0.1
    sortedcontainers          2.4.0
    soupsieve                 2.8
    sparkmagic                0.21.0
    Sphinx                    8.1.3
    sphinxcontrib-applehelp   2.0.0
    sphinxcontrib-devhelp     2.0.0
    sphinxcontrib-htmlhelp    2.1.0
    sphinxcontrib-jsmath      1.0.1
    sphinxcontrib-qthelp      2.0.0
    sphinxcontrib-serializinghtml 1.1.10
    sphinxcontrib-websupport  1.2.7
    spyder                    5.2.2
    spyder-kernels            2.2.1
    SQLAlchemy                2.0.44
    starlette                 0.49.3
    statsmodels               0.14.5
    sympy                     1.14.0
    tables                    3.10.1
    tblib                     3.1.0
    tenacity                  9.1.2
    terminado                 0.18.1
    testpath                  0.6.0
    text-unidecode            1.3
    textdistance              4.6.3
    threadpoolctl             3.6.0
    three-merge               0.1.1
    tifffile                  2020.6.3
    tinycss2                  1.4.0
    toml                      0.10.2
    tomli                     2.3.0
    tomlkit                   0.13.3
    toolz                     1.1.0
    tornado                   6.5.2
    tqdm                      4.67.1
    traitlets                 5.14.3
    typed-ast                 1.5.5
    typer                     0.20.0
    typing_extensions         4.15.0
    typing-inspection         0.4.2
    typing_utils              0.1.0
    tzdata                    2025.2
    ujson                     5.11.0
    unicodedata2              17.0.0
    uri-template              1.3.0
    urllib3                   2.5.0
    uvicorn                   0.38.0
    watchdog                  6.0.0
    wcwidth                   0.2.14
    webcolors                 25.10.0
    webencodings              0.5.1
    websocket-client          1.9.0
    Werkzeug                  3.1.3
    wheel                     0.45.1
    widgetsnbextension        4.0.15
    wrapt                     2.0.0
    wurlitzer                 3.1.1
    xlsxwriter                3.2.9
    xyzservices               2025.10.0
    yapf                      0.43.0
    yarl                      1.22.0
    zict                      3.0.0
    zipp                      3.23.0
    zope.event                6.0
    zope.interface            8.0.1
    zstandard                 0.25.0
    ```